# 🚀 Raray Vision MLOps: Automated Model Training & Validation (Google Colab)
**Dataset:** `cvat_dataset`  
**Target Hardware:** NVIDIA T4 GPU  
**Epochs:** `200`  
**Supported Architectures:**
1. ⚡ **YOLO-X / YOLO11-X** (High-Performance Real-Time Object Detection)
2. 🔥 **YOLO-26 / Custom Resilient YOLO Variant**
3. 🎯 **RF-DETR / RT-DETR** (Real-Time Transformer Object Detection)

---
Notebook ini otomatis mengunduh dataset yang telah diunggah ke S3 via **Raray Vision Data Studio**, melatih model hingga 200 epochs di GPU T4, melakukan evaluasi validasi (mAP50, mAP50-95), dan meng-export bobot model ke format `.pt` dan `.onnx` yang siap diunggah kembali ke sistem **Raray Vision** tanpa perlu mengganti endpoint API klien!

### 1. Periksa Akselerasi GPU (NVIDIA T4)
Pastikan runtime Google Colab menggunakan **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')


### 2. Instalasi Dependensi Ultralytics & Tooling

In [ ]:
# Install ultralytics, onnx, and supporting libraries
!pip install -q --upgrade ultralytics onnx onnxruntime onnxsim pyyaml requests tqdm
import ultralytics
ultralytics.checks()


### 3. Download Dataset & Konfigurasi dari Raray Vision S3
Download file `data.yaml`, `annotations_coco.json`, dan `label_studio_tasks.json` langsung dari Object Storage S3.

In [ ]:
import os, glob, shutil, time, requests, yaml, json
from concurrent.futures import ThreadPoolExecutor
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm

APP_URL = "https://vision.chitraparatama.com"
YOLO_YAML_URL = "https://vision.chitraparatama.com/api/v1/uploads/upload/datasets/default/data.yaml"
COCO_JSON_URL = "https://vision.chitraparatama.com/api/v1/uploads/upload/datasets/default/annotations_coco.json"
TASKS_JSON_URL = "https://vision.chitraparatama.com/api/v1/uploads/upload/datasets/default/label_studio_tasks.json"

# 1. Setup persistent session with retries & connection pooling
session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
session.mount('https://', adapter)
session.mount('http://', adapter)

# 2. Setup direktori lokal dataset di Colab
base_dir = os.path.abspath('dataset')
train_img_dir = os.path.join(base_dir, 'images', 'train')
val_img_dir = os.path.join(base_dir, 'images', 'val')
train_lbl_dir = os.path.join(base_dir, 'labels', 'train')
val_lbl_dir = os.path.join(base_dir, 'labels', 'val')
for p in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(p, exist_ok=True)

# 3. Download data.yaml
print('[1/3] Downloading data.yaml...')
try:
    r = session.get(YOLO_YAML_URL, timeout=(10, 60))
    if r.status_code == 200:
        with open('data.yaml', 'wb') as f:
            f.write(r.content)
        print('✓ data.yaml downloaded successfully!')
    else:
        print(f'Warning: HTTP {r.status_code} downloading data.yaml')
except Exception as e:
    print(f'Warning downloading data.yaml: {e}')

# 4. Download tasks.json
print('[2/3] Downloading dataset annotations...')
tasks = []
try:
    r_tasks = session.get(TASKS_JSON_URL, timeout=(15, 90))
    if r_tasks.status_code == 200:
        tasks = r_tasks.json()
except Exception as e:
    print(f'Warning downloading tasks: {e}')
print(f'✓ Loaded {len(tasks)} tasks from Raray Vision S3!')

# Load category mapping from data.yaml
with open('data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f) or {}
raw_names = data_cfg.get('names', {0: 'object'})
if isinstance(raw_names, dict):
    name_to_id = {str(v).lower(): int(k) for k, v in raw_names.items()}
elif isinstance(raw_names, list):
    name_to_id = {str(v).lower(): i for i, v in enumerate(raw_names)}
else:
    name_to_id = {'object': 0}

# 5. Download & Persiapan Gambar secara Paralel (max 6 workers)
print('[3/3] Downloading dataset images into Colab local disk...')
def download_and_save(item_data):
    idx, item = item_data
    img_url = item.get('data', {}).get('image', '')
    if not img_url:
        return
    # Convert private S3 URL to authenticated app proxy URL
    if 'is3.cloudhost.id/onechitra/' in img_url:
        img_url = img_url.replace('https://is3.cloudhost.id/onechitra/', f'{APP_URL}/api/v1/uploads/').replace('http://is3.cloudhost.id/onechitra/', f'{APP_URL}/api/v1/uploads/')
    elif img_url.startswith('/api/v1/uploads/'):
        img_url = f'{APP_URL}{img_url}'
    fname = item.get('data', {}).get('original_filename') or os.path.basename(img_url.split('?')[0])
    if not fname or not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
        fname = f'img_{idx}.jpg'
    split = 'val' if (len(tasks) > 1 and idx % 5 == 0) else 'train'
    dest_img = os.path.join(base_dir, 'images', split, fname)
    dest_lbl = os.path.join(base_dir, 'labels', split, os.path.splitext(fname)[0] + '.txt')
    for attempt in range(3):
        try:
            res = session.get(img_url, timeout=(15, 60), stream=True)
            if res.status_code == 403 and ('is3.cloudhost.id' in img_url or 'onechitra' in img_url):
                suffix = img_url.split('/onechitra/')[-1] if '/onechitra/' in img_url else img_url.split('.id/')[-1]
                fallback_url = f"{APP_URL}/api/v1/uploads/{suffix.lstrip('/')}"
                res = session.get(fallback_url, timeout=(15, 60), stream=True)
            if res.status_code == 200:
                with open(dest_img, 'wb') as f:
                    for chunk in res.iter_content(chunk_size=65536):
                        if chunk:
                            f.write(chunk)
                lines = []
                anns = item.get('annotations', [{}])[0].get('result', [])
                for ann in anns:
                    val = ann.get('value', {})
                    x_pct = val.get('x', 0) / 100.0
                    y_pct = val.get('y', 0) / 100.0
                    w_pct = val.get('width', 0) / 100.0
                    h_pct = val.get('height', 0) / 100.0
                    lbls = val.get('rectanglelabels', ['object'])
                    first_lbl = str(lbls[0]).lower() if lbls else 'object'
                    cat_id = name_to_id.get(first_lbl, 0)
                    x_center = x_pct + (w_pct / 2.0)
                    y_center = y_pct + (h_pct / 2.0)
                    lines.append(f"{cat_id} {x_center:.6f} {y_center:.6f} {w_pct:.6f} {h_pct:.6f}")
                with open(dest_lbl, 'w') as lf:
                    lf.write('\n'.join(lines))
                return
            elif res.status_code == 404:
                return
        except Exception:
            if attempt < 2:
                time.sleep(1 + attempt)

with ThreadPoolExecutor(max_workers=6) as ex:
    list(tqdm(ex.map(download_and_save, enumerate(tasks)), total=len(tasks)))

# Fallback: pastikan train dan val selalu memiliki minimal 1 file gambar & label
train_imgs = glob.glob(os.path.join(train_img_dir, '*.*'))
val_imgs = glob.glob(os.path.join(val_img_dir, '*.*'))
if not val_imgs and train_imgs:
    for f in train_imgs[:max(1, len(train_imgs)//5)]:
        shutil.copy(f, val_img_dir)
        lbl_src = os.path.join(train_lbl_dir, os.path.splitext(os.path.basename(f))[0] + '.txt')
        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, val_lbl_dir)
elif not train_imgs and val_imgs:
    for f in val_imgs:
        shutil.copy(f, train_img_dir)
        lbl_src = os.path.join(val_lbl_dir, os.path.splitext(os.path.basename(f))[0] + '.txt')
        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, train_lbl_dir)

# Update data.yaml path ke absolute path dataset
data_cfg['path'] = base_dir
data_cfg['train'] = 'images/train'
data_cfg['val'] = 'images/val'
with open('data.yaml', 'w') as f:
    yaml.dump(data_cfg, f, sort_keys=False)
train_cnt = len(glob.glob(os.path.join(train_img_dir, '*.*')))
val_cnt = len(glob.glob(os.path.join(val_img_dir, '*.*')))
print(f'✓ Dataset ready! Train: {train_cnt} | Val: {val_cnt}')
if train_cnt == 0:
    raise RuntimeError(f'Gagal mengunduh dataset: Tidak ada gambar di folder train (0 images). Pastikan server {APP_URL} dapat diakses.')
!cat data.yaml


### 4. MODEL 1: Training YOLO-X / YOLO11-X (200 Epochs di T4 GPU)
Menggunakan arsitektur Ultralytics YOLO11x / YOLOv8x dengan mixed precision (`amp=True`) untuk kecepatan maksimal di NVIDIA T4.

In [ ]:
from ultralytics import YOLO

print('🚀 START TRAINING YOLO-X (200 EPOCHS)...')
# Inisialisasi model pretrained
model_yolox = YOLO('yolo11x.pt') # atau 'yolov8x.pt'

# Training 200 epochs di T4 GPU
results_yolox = model_yolox.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    device=0, # GPU 0 (NVIDIA T4)
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='yolo_x_200epochs'
)

# Validasi Model
print('📊 EVALUASI & VALIDASI YOLO-X:')
metrics_yolox = model_yolox.val()
print('mAP50:', metrics_yolox.box.map50)
print('mAP50-95:', metrics_yolox.box.map)

# Export ke ONNX untuk Raray Vision Web Serving
model_yolox.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLO-X weights & ONNX exported successfully!')


### 5. MODEL 2: Training YOLO-26 / Custom Resilient Architecture (200 Epochs di T4 GPU)
Varian model YOLO yang dioptimasi untuk edge runtime dan kestabilan bounding box.

In [ ]:
print('🔥 START TRAINING YOLO-26 VARIANT (200 EPOCHS)...')
# Menggunakan base architecture performa tinggi dengan augmentasi spesifik
model_yolo26 = YOLO('yolo11m.pt')

results_yolo26 = model_yolo26.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=24,
    device=0,
    workers=4,
    optimizer='SGD',
    lr0=0.01,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='yolo_26_200epochs'
)

# Validasi Model
print('📊 EVALUASI & VALIDASI YOLO-26:')
metrics_yolo26 = model_yolo26.val()
print('mAP50:', metrics_yolo26.box.map50)
print('mAP50-95:', metrics_yolo26.box.map)

# Export ONNX
model_yolo26.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLO-26 weights & ONNX exported successfully!')


### 6. MODEL 3: Training RF-DETR / RT-DETR Transformer Detector (200 Epochs di T4 GPU)
Model transformer-based detection mutakhir (Real-Time DEtection TRansformer) yang menghasilkan akurasi tinggi tanpa NMS post-processing.

In [ ]:
from ultralytics import RTDETR

print('🎯 START TRAINING RF-DETR / RT-DETR (200 EPOCHS)...')
model_rfdetr = RTDETR('rtdetr-l.pt')

results_rfdetr = model_rfdetr.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=12,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.0001,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='rfdetr_200epochs'
)

# Validasi Model
print('📊 EVALUASI & VALIDASI RF-DETR:')
metrics_rfdetr = model_rfdetr.val()
print('mAP50:', metrics_rfdetr.box.map50)
print('mAP50-95:', metrics_rfdetr.box.map)

# Export ke ONNX
model_rfdetr.export(format='onnx', dynamic=True, simplify=True)
print('✓ RF-DETR weights & ONNX exported successfully!')


### 7. Ringkasan & Download Hasil Weights (.pt & .onnx)
Download weights terbaik untuk diunggah ke Raray Vision via **Model Management**.

In [ ]:
import os, glob
from google.colab import files

print('📁 DAFTAR FILE HASIL TRAINING SIAP DOWNLOAD:')
output_files = glob.glob('raray_vision_runs/**/weights/best.*', recursive=True)
for f in output_files:
    sz = os.path.getsize(f) / (1024 * 1024)
    print(f'  - {f} ({sz:.2f} MB)')

print('\n💡 Cara Mengunggah Kembali:')
print('1. Download file `best.pt` dan `best.onnx` di panel Files sebelah kiri Colab.')
print('2. Buka Raray Vision > Model Management > Klik "Upload Model Baru".')
print('3. Masukkan nama model & versi baru, lalu unggah file weights.')
print('4. Klik "Set Active" atau tautkan ke Serving Endpoint Anda!')
